# Joins

In this document im going to create the dims and fact tables from what i have in the silver layer (The big table).

#### Imports:

In [28]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

#### Database connection details:

In [29]:
try:
    pg_host = os.environ["POSTGRES_HOST"]
    pg_port = os.environ["POSTGRES_PORT"]
    pg_db = os.environ["POSTGRES_DB"]
    pg_user = os.environ["POSTGRES_USER"]
    pg_password = os.environ["POSTGRES_PASSWORD"]
    print(f"Connecting to database {pg_db} at {pg_host}:{pg_port} as user {pg_user}")
except KeyError as e:
    print(f"Environment variable not set: {e}")


Connecting to database PostgreSdB at localhost:5439 as user PostgreSuSer


#### We have to create the primary keys for the dims tables an indexes for the cols in  fact and them joining to reduce the Seq scanning and takes take more time

In [ ]:
pg_conn = psycopg2.connect(
    dbname=pg_db,
    user=pg_user,
    password=pg_password,
    host=pg_host,
    port=pg_port
)
cur = pg_conn.cursor()

try:
    cur.execute("ALTER TABLE data_warehouse.users_dim ADD PRIMARY KEY (user_key);")
    pg_conn.commit()

    cur.execute("ALTER TABLE data_warehouse.products_dim ADD PRIMARY KEY (product_key);")
    pg_conn.commit()

    cur.execute("""
        ALTER TABLE data_warehouse.sales_transactions_fact
            ADD CONSTRAINT fk_user FOREIGN KEY (user_key)
            REFERENCES data_warehouse.users_dim (user_key);
    """)
    pg_conn.commit()

    cur.execute("""
        ALTER TABLE data_warehouse.sales_transactions_fact
            ADD CONSTRAINT fk_product FOREIGN KEY (product_key)
            REFERENCES data_warehouse.products_dim (product_key);
    """)
    pg_conn.commit()

    cur.execute("CREATE INDEX IF NOT EXISTS idx_fact_user_key ON data_warehouse.sales_transactions_fact (user_key);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_fact_product_key ON data_warehouse.sales_transactions_fact (product_key);")
    pg_conn.commit()

    cur.execute("""
        SELECT
            u.user_id,
            p.product_id,
            p.brand,
            f.price,
            f.event_time
        FROM data_warehouse.sales_transactions_fact f
        INNER JOIN data_warehouse.users_dim u ON f.user_key = u.user_key
        INNER JOIN data_warehouse.products_dim p ON f.product_key = p.product_key
        LIMIT 10;
    """)

    print("Joins Created Successfully.")

except Exception as e:
    pg_conn.rollback()
    print(f"Error: {e}")

[('561409814', '3600545', 'samsung', Decimal('579.140'), datetime.datetime(2019, 11, 1, 6, 9, 11)), ('565123263', '12719633', 'kapsen', Decimal('60.230'), datetime.datetime(2019, 11, 1, 5, 28, 13)), ('566056042', '5701166', '', Decimal('144.120'), datetime.datetime(2019, 11, 1, 7, 16, 38)), ('566222589', '12700704', 'matador', Decimal('80.570'), datetime.datetime(2019, 11, 1, 10, 44, 20)), ('566319273', '1005144', 'apple', Decimal('1658.190'), datetime.datetime(2019, 11, 1, 6, 39, 58)), ('566331822', '1004888', 'samsung', Decimal('224.460'), datetime.datetime(2019, 11, 1, 5, 49, 1)), ('566332401', '1004566', 'huawei', Decimal('164.840'), datetime.datetime(2019, 11, 1, 5, 57, 7)), ('566332401', '1004566', 'huawei', Decimal('164.840'), datetime.datetime(2019, 11, 1, 5, 58, 49)), ('566347725', '1004836', 'samsung', Decimal('229.900'), datetime.datetime(2019, 11, 1, 6, 49, 15)), ('566347725', '1004873', 'samsung', Decimal('362.100'), datetime.datetime(2019, 11, 1, 6, 48, 15))]
Done.
